# Load the checkpoint

In [ ]:
import os
import sys

import dill
import hydra
import numpy as np
import PyriteUtility
import torch
from torch.utils.data import DataLoader

from diffusion_policy.dataset.base_dataset import BaseDataset, BaseImageDataset
from diffusion_policy.workspace.base_workspace import BaseWorkspace

print(sys.executable)


print(PyriteUtility.__file__)


print(PyriteUtility.__path__)


ckpt_path = "/store/real/hjchoi92/training_outputs/pods/UMIFT-RGB-DEPTH-AH_5_33-DS8-AP-425_WBW2-plain-add-wO/checkpoints/latest.ckpt"
# ckpt_path = data_path + "2025.05.01_20.16.02_umi-ft-single-arm-two-coinft-AH_8_WBW-iph/checkpoints/latest.ckpt"

device = torch.device("cpu")

# load checkpoint
if not ckpt_path.endswith(".ckpt"):
    ckpt_path = os.path.join(ckpt_path, "checkpoints", "latest.ckpt")
payload = torch.load(open(ckpt_path, "rb"), map_location="cpu", pickle_module=dill)
cfg = payload["cfg"]

# print(cfg.policy.obs_encoder.keys())
# print(cfg.policy.obs_encoder.force_encoder_cfg.keys())

print(
    "vision_encoder_model_name:", cfg.policy.obs_encoder.vision_encoder_cfg.model_name
)
print("force_encoder_model_name:", cfg.policy.obs_encoder.force_encoder_cfg.model_name)
print("dataset_path:", cfg.task.dataset.dataset_path)
print("Dataset path exists:", os.path.exists(cfg.task.dataset.dataset_path))

cls = hydra.utils.get_class(cfg._target_)
workspace = cls(cfg)
workspace: BaseWorkspace
workspace.load_payload(payload, exclude_keys=None, include_keys=None)

policy = workspace.model
if cfg.training.use_ema:
    policy = workspace.ema_model
policy.num_inference_steps = cfg.policy.num_inference_steps  # DDIM inference iterations

policy.eval().to(device)
policy.reset()

# use normalizer saved in the policy
sparse_normalizer, dense_normalizer = policy.get_normalizer()

shape_meta = cfg.task.shape_meta

print("Sparse action horizon:", cfg.task.sparse_action_horizon)

# Load a dataset

In [ ]:
from hydra import compose, initialize
from omegaconf import OmegaConf

# # load the dataset used in training
# dataset: BaseImageDataset
# dataset = hydra.utils.instantiate(cfg.task.dataset)
# assert isinstance(dataset, BaseImageDataset) or isinstance(dataset, BaseDataset)
# print("Test Script: Creating dataloader.")
# train_dataloader = DataLoader(dataset, **cfg.dataloader)
# print('train dataset:', len(dataset), 'train dataloader:', len(train_dataloader))

# load the dataset specified in config

with initialize(
    version_base=None,
    config_path=str("../diffusion_policy/config"),
    job_name="test_app",
):
    cfg = compose(config_name="train_conv_workspace")
    OmegaConf.resolve(cfg)

    print("Test Script: configuring dataset.")
    dataset: BaseImageDataset
    dataset = hydra.utils.instantiate(cfg.task.dataset)
    assert isinstance(dataset, BaseImageDataset) or isinstance(dataset, BaseDataset)
    print("Test Script: Creating dataloader.")
    train_dataloader = DataLoader(dataset, **cfg.dataloader)
    print("train dataset:", len(dataset), "train dataloader:", len(train_dataloader))

# Run some tests

In [ ]:
import torch.nn.functional as F
from einops import reduce


def log_action_mse(step_log, category, pred_action, gt_action):
    pred_naction = {
        "sparse": sparse_normalizer["action"].normalize(pred_action["sparse"]),
        # 'dense': dense_normalizer['action'].normalize(pred_action['dense'])
    }
    gt_naction = {
        "sparse": sparse_normalizer["action"].normalize(gt_action["sparse"]),
        # 'dense': dense_normalizer['action'].normalize(gt_action['dense'])
    }

    B, T, _ = pred_naction["sparse"].shape
    pred_naction_sparse = pred_naction["sparse"].view(B, T, -1, 9)
    gt_naction_sparse = gt_naction["sparse"].view(B, T, -1, 9)
    sparse_loss = F.mse_loss(pred_naction_sparse, gt_naction_sparse, reduction="none")
    sparse_loss = sparse_loss.type(sparse_loss.dtype)
    sparse_loss = reduce(sparse_loss, "b ... -> b (...)", "mean")
    sparse_loss = sparse_loss.mean()

    step_log[f"{category}_sparse_naction_mse_error"] = float(sparse_loss.detach())
    # step_log[f'{category}_sparse_naction_mse_error_pos'] = F.mse_loss(pred_naction_sparse[..., :3], gt_naction_sparse[..., :3])
    # step_log[f'{category}_sparse_naction_mse_error_rot'] = F.mse_loss(pred_naction_sparse[..., 3:9], gt_naction_sparse[..., 3:9])
    B, T, _, _ = pred_naction["dense"].shape
    pred_naction_dense = pred_naction["dense"].view(B, T, -1, 9)
    gt_naction_dense = gt_naction["dense"].view(B, T, -1, 9)
    dense_loss = F.mse_loss(pred_naction_dense, gt_naction_dense, reduction="none")
    dense_loss = dense_loss.type(dense_loss.dtype)
    dense_loss = reduce(dense_loss, "b ... -> b (...)", "mean")
    dense_loss = dense_loss.mean()
    step_log[f"{category}_dense_naction_mse_error"] = float(dense_loss.detach())
    # step_log[f'{category}_dense_naction_mse_error_pos'] = F.mse_loss(pred_naction_dense[..., :3], gt_naction_dense[..., :3])
    # step_log[f'{category}_dense_naction_mse_error_rot'] = F.mse_loss(pred_naction_dense[..., 3:9], gt_naction_dense[..., 3:9])


# get a batch of data'
print("get a batch of data")
batch = next(iter(train_dataloader))

print(batch.keys())
for key, attr in batch["obs"]["sparse"].items():
    print("   obs.sparse.key: ", key, attr.shape)
for key, attr in batch["obs"]["dense"].items():
    print("   obs.dense.key: ", key, attr.shape)

# print(batch['action'].items())
# for key, attr in batch['action'].items():
#     print("   action.key: ", key, attr.shape)

In [ ]:
import os

import matplotlib.pyplot as plt
import torch

# Ensure save directory exists
save_dir = os.getcwd()  # Current working directory
print(f"Saving visualizations to: {save_dir}")

anchor = 45  # select which sample of the 128 in the batch to visualize
num_samples = 10  # Number of samples to visualize

for key, attr in batch["obs"]["sparse"].items():
    print(f"Processing: {key} | Shape: {attr.shape}")

    # Extract first `num_samples` from batch
    samples = (
        attr[anchor : anchor + num_samples].cpu().numpy()
    )  # Convert tensor to numpy

    if len(samples.shape) == 5:  # (batch, time, channel, height, width)
        for i in range(num_samples):
            img = samples[
                i, 0
            ]  # Take the first frame from the two sequence (time index 0)
            img = np.transpose(img, (1, 2, 0))  # Convert (C, H, W) -> (H, W, C)
            img_path = os.path.join(save_dir, f"{key}_sample_{i}.png")

            if key == "depth_0":
                depth_min = np.min(img[35:190, :, :])
                depth_max = np.max(img[35:190, :, :])
                plt.imsave(img_path, img, vmin=0.0, vmax=1.0)
                print(f"Sample {i} Depth min: {depth_min}, Depth max: {depth_max}")
            else:
                rgb_min = np.min(img[35:190, :, :])
                rgb_max = np.max(img[35:190, :, :])
                plt.imsave(img_path, img)
                print(f"Sample {i} rgb min: {rgb_min}, rgb max: {rgb_max}")
                # print(f"Saved image: {img_path}")

    else:  # If it's a numeric tensor (e.g., positions, forces)
        for i in range(num_samples):
            sample_data = samples[i]  # Extract one sample

            # If time dimension exists, plot it over time
            if len(sample_data.shape) == 2:  # (T, D) format
                plt.figure(figsize=(8, 4))
                for j in range(sample_data.shape[1]):  # Plot each dimension
                    plt.plot(
                        sample_data[:, j],
                        marker="o",
                        linestyle="-",
                        label=f"Dim {j + 1}",
                    )

                plt.xlabel("Time step")
                plt.ylabel(key)
                plt.title(f"{key} Sample {i}")
                plt.legend()
                plt.grid(True)

                # Save plot
                plot_path = os.path.join(save_dir, f"{key}_sample_{i}.png")
                plt.savefig(plot_path)
                plt.close()
                # print(f"Saved plot: {plot_path}")

In [ ]:
# test compute loss
print("running policy on batch")
flag = {"dense_traj_cond_use_gt": True}
raw_loss = policy(batch, flag)
print("total loss: ", raw_loss)
print("sparse loss:", policy.sparse_loss)
print("dense loss:", policy.dense_loss)

print("trying to predict action")

# test predict action
gt_action = batch["action"]

pred_action = policy.predict_action(
    batch["obs"], batch["action"]
)  # providing batch will enable gt sparse condition
print("gt_action['sparse'].shape: ", gt_action["sparse"].shape)
print("pred_action['sparse'].shape: ", pred_action["sparse"].shape)
# print("gt_action['dense'].shape: ", gt_action['dense'].shape)
# print("pred_action['dense'].shape: ", pred_action['dense'].shape)

# step_log = {}
# log_action_mse(step_log, 'train', pred_action, gt_action)
# print(json.dumps(step_log, indent=4))

In [ ]:
print(type(policy))
print(batch["obs"].keys())
print(batch["obs"]["sparse"].keys())
print(batch["action"].keys())
print(batch["action"]["sparse"].shape)
print(batch["action"]["dense"].keys())
print(pred_action.keys())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

# Choose 3 dimensions (modify as needed)
r_pose0, r_pose1, r_pose2 = 0, 1, 2  # reference position - actual robot position
v_pose0, v_pose1, v_pose2 = 9, 10, 11  # virtual target position
gripper = 19  # gripper position
grasp_force = 20

# anchor = 50 # select which sample of the 128 in the batch to visualize
# subsample_factor = 10


# Convert to numpy for plotting
gt_data = gt_action["sparse"].cpu().detach().numpy()  # shape: (128, 20, 21)
pred_data = pred_action["sparse"].cpu().detach().numpy()  # shape: (128, 20, 21)


for ns in range(num_samples):
    gt_r_pose = gt_data[
        anchor + ns, :, [r_pose0, r_pose1, r_pose2]
    ].T  # converting (3, 20) to (30, 3)...
    pred_r_pose = pred_data[anchor + ns, :, [r_pose0, r_pose1, r_pose2]].T

    gt_v_pose = gt_data[anchor + ns, :, [v_pose0, v_pose1, v_pose2]].T
    pred_v_pose = pred_data[anchor + ns, :, [v_pose0, v_pose1, v_pose2]].T

    gt_gripper = gt_data[anchor + ns, :, gripper]
    pred_gripper = pred_data[anchor + ns, :, gripper]

    gt_grasp_force = gt_data[anchor + ns, :, grasp_force]
    pred_grasp_force = pred_data[anchor + ns, :, grasp_force]

    # If time dimension exists, plot it over time
    plt.figure(figsize=(8, 4))
    plt.plot(gt_gripper[ns], marker="o", linestyle="-", c="blue", label="Ground Truth")
    plt.plot(pred_gripper[ns], marker="o", linestyle="-", c="green", label="Prediction")

    plt.xlabel("Time step")
    plt.ylabel("gripper")
    plt.title(f"gripper Sample {ns}")
    plt.legend()
    plt.grid(True)

    # Save plot
    plot_path = os.path.join(save_dir, f"gripper_sample_{ns}.png")
    plt.savefig(plot_path)
    plt.show()
    print(f"Saved plot: {plot_path}")

    # If time dimension exists, plot it over time
    plt.figure(figsize=(8, 4))
    plt.plot(
        gt_grasp_force[ns], marker="o", linestyle="-", c="blue", label="Ground Truth"
    )
    plt.plot(
        pred_grasp_force[ns], marker="o", linestyle="-", c="green", label="Prediction"
    )

    plt.xlabel("Time step")
    plt.ylabel("grasp force")
    plt.title(f"grasp force Sample {ns}")
    plt.legend()
    plt.grid(True)

    # Save plot
    plot_path = os.path.join(save_dir, f"grasp_force_sample_{ns}.png")
    plt.savefig(plot_path)
    plt.show()
    print(f"Saved plot: {plot_path}")

    # Create 3D plot
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111)
    ax.scatter(gt_grasp_force, pred_grasp_force, c="blue", label="", alpha=0.6)
    plt.show()

    # Create 3D plot
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection="3d")
    # ax = fig.add_subplot(111)

    # Plot ground truth (blue) and predicted actions (red)
    ax.scatter(
        gt_r_pose[:, 0],
        gt_r_pose[:, 1],
        gt_r_pose[:, 2],
        c="blue",
        label="Ground Truth Ref",
        alpha=0.6,
    )
    # ax.scatter(gt_v_pose[:, 0], gt_v_pose[:, 1], gt_v_pose[:, 2], c='red', label='Ground Truth Virtual', alpha=0.6)
    print("gt_r_pose.shape: ", gt_r_pose.shape)

    ax.scatter(
        pred_r_pose[:, 0],
        pred_r_pose[:, 1],
        pred_r_pose[:, 2],
        c="green",
        label="Prediction Ref",
        alpha=0.6,
    )
    # ax.scatter(pred_v_pose[:, 0], pred_v_pose[:, 1], pred_v_pose[:, 2], c='purple', label='Prediction Virtual', alpha=0.6)
    print("pred_r_pose.shape: ", pred_r_pose.shape)

    # ax.scatter(gt_points[:, 0], gt_points[:, 1], c='blue', label='Ground Truth', alpha=0.6)
    # ax.scatter(pred_points[:, 0], pred_points[:, 1], c='red', label='Prediction', alpha=0.6)

    # Labels and legend
    ax.set_xlabel("Dimension 1")
    ax.set_ylabel("Dimension 2")
    ax.set_zlabel("Dimension 3")
    ax.set_title("3D Plot of Actions")
    ax.legend()

    # Save the plot
    plot_filename = f"action_3d_plot_{ns}.png"
    plt.savefig(plot_filename, dpi=300)
    plt.show()

    print(f"Plot saved as {plot_filename}")

Palette.. check recorded horizon data:

In [ ]:
import os
import shutil

import numpy as np
import zarr

# Define source directory and output Zarr file
source_dir = "/store/real/chuerpan/umift_playback"
output_zarr_path = "/store/real/hjchoi92/data/real_processed/umift/WBW_playback.zarr"  # Single Zarr file

# Remove existing Zarr file if needed
if os.path.exists(output_zarr_path):
    shutil.rmtree(output_zarr_path)

# Create a single Zarr store
zarr_root = zarr.open(output_zarr_path, mode="w")

# Mapping horizons to episode names
horizon_to_episode = {
    "horizon_0": "episode_0",
    "horizon_1": "episode_1",
    "horizon_2": "episode_2",
}

# Convert and store all horizons inside the single Zarr file
for horizon, episode in horizon_to_episode.items():
    horizon_path = os.path.join(source_dir, horizon)

    episode_group = zarr_root.require_group(f"data/{episode}")

    # Copy each key in the horizon directory into Zarr
    for key in os.listdir(horizon_path):
        key_path = os.path.join(horizon_path, key)
        if os.path.isdir(key_path):  # Ensure it's a directory containing data
            for sub_key in os.listdir(key_path):
                sub_key_path = os.path.join(key_path, sub_key)

                # Ensure file is a NumPy array (.npy)
                if sub_key.endswith(".npy"):
                    data = np.load(sub_key_path)  # Load NumPy array
                    episode_group[f"{key}/{sub_key.replace('.npy', '')}"] = (
                        data  # Store in Zarr
                    )

    print(f"Stored {horizon} as {episode} in {output_zarr_path}")

print("Conversion to single Zarr file completed!")

# Print structure of the final merged dataset
print("Final Zarr Structure:")
print(zarr_root.tree())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import zarr

# Path to collected data
robot_data_path = "/store/real/chuerpan/umift_playback"  # Update if needed
robot_root = zarr.open(robot_data_path, mode="r")

# Select which horizon to visualize
horizon = "horizon_0"  # Change to "horizon_1" or "horizon_2" if needed

# Print full dataset structure
print(f"Structure of {horizon}:")
print(robot_root[horizon].tree())


# Alternatively, list all dataset keys manually
def list_all_keys(node, prefix=""):
    """Recursively print all keys in a Zarr dataset."""
    for key in node.keys():
        full_key = f"{prefix}/{key}" if prefix else key
        print(full_key)
        if isinstance(node[key], zarr.Group):  # If it's a group, recurse
            list_all_keys(node[key], full_key)


print("\nListing all keys in the dataset:")
list_all_keys(robot_root[horizon])

# Extract observation data
obs = robot_root[horizon]["obs"]

# Visualize RGB Image
if "rgb_0" in obs:
    rgb_data = np.array(obs["rgb_0"])
    print(f"RGB Data Shape: {rgb_data.shape}")  # Expected: (T, H, W, C)

    plt.figure(figsize=(5, 5))
    plt.imshow(rgb_data[0])  # Show the first frame
    plt.title(f"{horizon} - RGB Frame 0")
    plt.axis("off")
    plt.show()

# Plot End-Effector Position over Time
if "robot0_eef_pos" in obs:
    eef_pos = np.array(obs["robot0_eef_pos"])
    print(f"EEF Position Shape: {eef_pos.shape}")  # Expected: (T, 3)

    plt.figure(figsize=(8, 4))
    plt.plot(eef_pos[:, 0], label="X")
    plt.plot(eef_pos[:, 1], label="Y")
    plt.plot(eef_pos[:, 2], label="Z")
    plt.legend()
    plt.xlabel("Time Step")
    plt.ylabel("End-Effector Position")
    plt.title(f"{horizon} - End-Effector Position")
    plt.grid(True)
    plt.show()

# Plot Gripper State
if "robot0_gripper" in obs:
    gripper_state = np.array(obs["robot0_gripper"])
    print(f"Gripper Data Shape: {gripper_state.shape}")  # Expected: (T, 1)

    plt.figure(figsize=(8, 4))
    plt.plot(gripper_state, marker="o")
    plt.xlabel("Time Step")
    plt.ylabel("Gripper State")
    plt.title(f"{horizon} - Gripper State")
    plt.grid(True)
    plt.show()

# Plot Wrench Data (Forces and Torques)
if "robot0_eef_wrench_left" in obs:
    wrench_left = np.array(obs["robot0_eef_wrench_left"])
    print(f"Wrench Left Shape: {wrench_left.shape}")  # Expected: (T, 6)

    plt.figure(figsize=(8, 4))
    for i in range(6):
        plt.plot(wrench_left[:, i], label=f"Dim {i + 1}")
    plt.xlabel("Time Step")
    plt.ylabel("Force/Torque")
    plt.title(f"{horizon} - Left Wrench")
    plt.legend()
    plt.grid(True)
    plt.show()